In [1]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define transforms - what happens to each image before it enters the model
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),        # Resize all images to 224x224
    transforms.RandomHorizontalFlip(),    # Randomly flip images - data augmentation
    transforms.RandomRotation(10),        # Randomly rotate up to 10 degrees
    transforms.ToTensor(),                # Convert image to tensor (0-1 range)
    transforms.Normalize(                 # Normalize using ImageNet mean/std
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),        # Same size as training
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transforms defined successfully")

Transforms defined successfully


In [2]:
# Point to your dataset
base = r"C:\Users\ragha\Desktop\manufacturing-defect-detection\data\raw\archive\casting_data\casting_data"

# Load datasets using ImageFolder
# ImageFolder automatically assigns labels based on folder names
# def_front = class 0, ok_front = class 1
train_dataset = datasets.ImageFolder(
    root=base + r"\train",
    transform=train_transforms
)

test_dataset = datasets.ImageFolder(
    root=base + r"\test",
    transform=test_transforms
)

# Create DataLoaders - these feed batches of images to the model
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")
print(f"Class mapping: {train_dataset.class_to_idx}")

Training samples: 6633
Test samples: 715
Classes: ['def_front', 'ok_front']
Class mapping: {'def_front': 0, 'ok_front': 1}


In [3]:
import torch.nn as nn

class BaseCNN(nn.Module):
    def __init__(self):
        super(BaseCNN, self).__init__()
        
        # Feature extraction layers - these learn to detect patterns
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # 3 input channels (RGB), 32 filters
            nn.ReLU(),                                    # Activation function
            nn.MaxPool2d(2, 2),                           # Downsample by half: 224 -> 112
            
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 32 in, 64 filters
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # 112 -> 56
            
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1), # 64 in, 128 filters
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                            # 56 -> 28
        )
        
        # Classification layers - take features and output a class
        self.classifier = nn.Sequential(
            nn.Flatten(),                        # Flatten 3D tensor to 1D
            nn.Linear(128 * 28 * 28, 256),       # Fully connected layer
            nn.ReLU(),
            nn.Dropout(0.5),                     # Randomly drop 50% of neurons during training
            nn.Linear(256, 2)                    # Output 2 values: one per class
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Create the model
model = BaseCNN()
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

BaseCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=100352, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=256, out_features=2, bias=True)
  )
)

Total parameters: 25,784,130


In [4]:
import torch.optim as optim

# Loss function - measures how wrong the model's predictions are
# CrossEntropyLoss is standard for classification
criterion = nn.CrossEntropyLoss()

# Optimizer - adjusts model weights to reduce loss
# Adam is a good default optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# Move model to device
model = model.to(device)

Training on: cpu


In [6]:
def train_model(model, train_loader, test_loader, criterion, optimizer, epochs=10):
    
    train_losses, test_losses, train_accs, test_accs = [], [], [], []
    
    for epoch in range(epochs):
        # --- Training phase ---
        model.train()  # Set model to training mode (enables dropout)
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()        # Clear gradients from last step
            outputs = model(images)      # Forward pass: get predictions
            loss = criterion(outputs, labels)  # Calculate loss
            loss.backward()              # Backward pass: calculate gradients
            optimizer.step()             # Update weights
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_loss = running_loss / len(train_loader)
        train_acc = 100. * correct / total
        
        # --- Evaluation phase ---
        model.eval()  # Set model to eval mode (disables dropout)
        running_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():  # Don't calculate gradients during evaluation
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                running_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        
        test_loss = running_loss / len(test_loader)
        test_acc = 100. * correct / total
        
        train_losses.append(train_loss)
        test_losses.append(test_loss)
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        
        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
              f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")
    
    return train_losses, test_losses, train_accs, test_accs

print("Training function defined")

Training function defined
